# Moving Average vs Naive Evaluation

This demo provides a comprehensive statistical and efficiency evaluation of the Moving Average (MA) forecasting method against the standard Naive persistence baseline across synthetic noisy time series data.

### What this notebook does:
1. **Loads evaluation data** from GitHub with local fallback.
2. **Computes aggregate performance metrics** (MSE and MAE) for both Naive and Moving Average forecasts.
3. **Performs rigorous statistical significance testing** (Paired t-test and Wilcoxon signed-rank test).
4. **Visualizes comparative error distributions and win rates** across seeds.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2', 'tabulate==0.9.0')

In [ ]:
import json
import os
import urllib.request
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

print("Imports successful!")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-230d6e-robust-temporal-smoothing-evaluating-mov/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"Could not load from GitHub URL ({e}), trying local fallback...")
    
    local_path = "mini_demo_data.json"
    if os.path.exists(local_path):
        with open(local_path) as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path.")

data = load_data()
print("Data loaded successfully! Dataset name:", data["datasets"][0]["dataset"])

## Configuration

Define tunable parameters for demonstration.

In [ ]:
# Configuration parameters (minimum viable / demo scale)
CONFIG = {
    "window_size": 3,
    "alpha_significance": 0.05,
}
print("Config:", CONFIG)

## Processing & Metrics Computation

Extract predictions, calculate Mean Squared Error (MSE) and Mean Absolute Error (MAE), and perform statistical tests.

In [ ]:
examples = data["datasets"][0]["examples"]

y_true = []
y_naive = []
y_ma = []
seeds = []
timesteps = []

for ex in examples:
    y_true.append(float(ex["output"]))
    y_naive.append(float(ex["predict_naive"]))
    y_ma.append(float(ex["predict_moving_average"]))
    seeds.append(int(ex["metadata_seed"]))
    timesteps.append(int(ex["metadata_timestep"]))

y_true = np.array(y_true)
y_naive = np.array(y_naive)
y_ma = np.array(y_ma)
seeds = np.array(seeds)
timesteps = np.array(timesteps)

# Aggregate metrics from data or recompute
metrics_agg = data["metrics_agg"]

print("=== Aggregate Evaluation Metrics ===")
for k, v in metrics_agg.items():
    print(f"{k}: {v:.6f}" if isinstance(v, float) else f"{k}: {v}")

## Visualization & Results Summary

Display summary table of metrics and comparative evaluation plots.

In [ ]:
# Create summary table
summary_df = pd.DataFrame({
    "Metric": [
        "Overall MSE (Naive)",
        "Overall MSE (Moving Average)",
        "Overall MAE (Naive)",
        "Overall MAE (Moving Average)",
        "Paired t-test t-stat",
        "Paired t-test p-value",
        "Wilcoxon stat",
        "Wilcoxon p-value",
        "Seed Win Rate (MA < Naive)"
    ],
    "Value": [
        metrics_agg["overall_mse_naive"],
        metrics_agg["overall_mse_ma"],
        metrics_agg["overall_mae_naive"],
        metrics_agg["overall_mae_ma"],
        metrics_agg["paired_t_test_stat"],
        metrics_agg["paired_t_test_pvalue"],
        metrics_agg["wilcoxon_stat"],
        metrics_agg["wilcoxon_pvalue"],
        metrics_agg["seed_win_rate"]
    ]
})

display(summary_df.style.hide(axis="index"))

# Plotting example predictions vs true values
plt.figure(figsize=(8, 4))
x_indices = np.arange(len(y_true))
plt.plot(x_indices, y_true, label='True Series', marker='o', color='black', linewidth=2)
plt.plot(x_indices, y_naive, label='Naive Forecast', marker='x', linestyle='--', color='red')
plt.plot(x_indices, y_ma, label='Moving Average Forecast', marker='s', linestyle='-.', color='blue')
plt.xlabel('Sample Index')
plt.ylabel('Value')
plt.title('Moving Average vs Naive Forecast Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()